# ETL — Propiedades Medellín
### Extracción · Transformación · Carga Simulada

Reglas aplicadas según hallazgos del EDA. Cada sección referencia la decisión de negocio que la justifica.

In [1]:
import pandas as pd
import re
import unicodedata
import math
from datetime import datetime
import os

CSV_PATH = os.getenv("CSV_PATH", "propiedades_medellin_raw.csv")

df_raw = pd.read_csv(CSV_PATH, encoding="utf-8-sig")
print(f"Filas    : {df_raw.shape[0]:,}")
print(f"Columnas : {df_raw.shape[1]}")

Filas    : 1,240
Columnas : 8


In [2]:
df = df_raw.copy()

In [3]:
print(f'Registros raw    : {len(df):,}')
print(f'IDs únicos       : {df["id_propiedad"].nunique():,}')
print(f'Duplicados       : {df.duplicated().sum()}')

Registros raw    : 1,240
IDs únicos       : 1,200
Duplicados       : 40


In [4]:
# La eliminación de duplicados NO se hace aquí sobre df completo.
# El CSV mezcla dos entidades con reglas de unicidad distintas:
#   propiedad → unicidad por id_propiedad (atributos físicos estables)
#   anuncio   → unicidad por (id_propiedad, fecha_publicacion) (publicación comercial)
# Se aplica al construir cada tabla más adelante.

In [5]:
print("Duplicados exactos (id + precio):", df.duplicated(subset=["id_propiedad", "precio_venta"]).sum())
print("Duplicados solo por id          :", df.duplicated(subset=["id_propiedad"]).sum())
print("Mismo id, precio distinto       :",
      df.duplicated(subset=["id_propiedad"]).sum() - df.duplicated(subset=["id_propiedad","precio_venta"]).sum())

Duplicados exactos (id + precio): 40
Duplicados solo por id          : 40
Mismo id, precio distinto       : 0


In [6]:
TIPO_MAPPING = {
    "casa":         "Casa",
    "apartamento":  "Apartamento",
    "apartestudio": "Apartestudio",
    "finca":        "Finca",
}

def limpiar_tipo(val):
    if pd.isna(val) or str(val).strip() in ("", "?"):
        return "Sin clasificar"
    return TIPO_MAPPING.get(str(val).strip().lower(), "Sin clasificar")

df["tipo_inmueble_clean"] = df["tipo_inmueble"].apply(limpiar_tipo)

print(df["tipo_inmueble_clean"].value_counts().to_string())


tipo_inmueble_clean
Finca             254
Apartamento       251
Casa              249
Sin clasificar    245
Apartestudio      241


In [7]:
ZONA_MAPPING = {
    # El Poblado
    "el poblado": 1, "poblado": 1,
    # Laureles
    "laureles": 2, "laureles - estadio": 2, "laureles estadio": 2,
    # Belén
    "belen": 3, "belén": 3,
    # Robledo
    "robledo": 4,
    # Centro
    "centro": 5, "centro - medellin": 5, "centro - medellín": 5,
    # El Hueco
    "el hueco": 6,
    # Sabaneta
    "sabaneta": 7,
    # Envigado
    "envigado": 8,
}


In [8]:
def normalizar(texto):
    """Lowercase + strip + colapsa espacios."""
    return re.sub(r"\s+", " ", str(texto).strip().lower())

def quitar_tildes(texto):
    return "".join(
        c for c in unicodedata.normalize("NFD", texto)
        if unicodedata.category(c) != "Mn"
    )

In [9]:
def limpiar_zona(val):
    if pd.isna(val) or str(val).strip() == "":
        return None
    norm = normalizar(val)
    if norm in ZONA_MAPPING:
        return ZONA_MAPPING[norm]
    sin_tildes = quitar_tildes(norm)
    if sin_tildes in ZONA_MAPPING:
        return ZONA_MAPPING[sin_tildes]
    # coincidencia parcial
    for clave, id_zona in ZONA_MAPPING.items():
        if quitar_tildes(clave) in sin_tildes or sin_tildes in quitar_tildes(clave):
            return id_zona
    return None

In [10]:
df["id_zona"] = df["ubicacion"].apply(limpiar_zona)

no_mapeadas = df["id_zona"].isna().sum()
print(f"Mapeadas    : {df['id_zona'].notna().sum()}")
print(f"Sin mapear  : {no_mapeadas}")
if no_mapeadas > 0:
    print("Valores no mapeados — estas propiedades serán rechazadas:")
    print(df[df["id_zona"].isna()]["ubicacion"].value_counts().to_string())

Mapeadas    : 1240
Sin mapear  : 0


In [11]:
# Estratos 1-6 — el id es el mismo número del estrato
# id 7 = "Sin estrato"

def limpiar_estrato(val):
    try:
        numero = int(float(val))
        if 1 <= numero <= 6:
            return numero
        return 7
    except (ValueError, TypeError):
        return 7

df["id_estrato"] = df["estrato_socioeconomico"].apply(limpiar_estrato)

print(df["id_estrato"].value_counts().sort_index().to_string())
print(f"\nSin estrato (id=7): {(df['id_estrato'] == 7).sum()}")

id_estrato
2    239
3    218
4    213
5    214
6    229
7    127

Sin estrato (id=7): 127


In [12]:
FORMATOS_FECHA = [
    "%b %d, %Y",   # Oct 15, 2023
    "%d/%m/%Y",    # 15/10/2023
    "%Y.%m.%d",    # 2023.10.15
    "%Y-%m-%d",    # 2023-10-15
]

def limpiar_fecha(val):
    if pd.isna(val) or str(val).strip() == "":
        return None
    for fmt in FORMATOS_FECHA:
        try:
            return datetime.strptime(str(val).strip(), fmt).strftime("%Y-%m-%d")
        except ValueError:
            continue
    return None  # formato desconocido — anuncio se descarta

df["fecha_clean"] = df["fecha_publicacion"].apply(limpiar_fecha)

# Distinguir campo vacío de formato no reconocido
sin_fecha_vacio   = df["fecha_publicacion"].isna().sum()
sin_fecha_formato = df["fecha_clean"].isna().sum() - sin_fecha_vacio
print(f"Fechas parseadas         : {df['fecha_clean'].notna().sum():,}")
print(f"Sin fecha (campo vacío)  : {sin_fecha_vacio}")
print(f"Sin fecha (fmt desconocido): {sin_fecha_formato}")

Fechas parseadas         : 1,240
Sin fecha (campo vacío)  : 0
Sin fecha (fmt desconocido): 0


In [13]:
# Regla: metraje <= 1 → NULL
# El EDA mostró que en el rango 1-20 m² solo existe el valor 1.
# No hay valores entre 2 y 20 — el umbral es exacto, no arbitrario.
# Valores negativos y cero también son imposibles físicamente.
def limpiar_metraje(val):
    try:
        v = float(val)
        return v if v > 1 else None
    except (ValueError, TypeError):
        return None

df["metraje_clean"] = df["metraje_m2"].apply(limpiar_metraje)

print(f"Metraje válido  : {df['metraje_clean'].notna().sum():,}")
print(f"Convertidos NULL: {df['metraje_clean'].isna().sum()} (incluye <= 1)")

Metraje válido  : 1,178
Convertidos NULL: 62 (incluye <= 1)


In [14]:
#Limpiar formato antes de castear.
# El EDA detectó 357 valores con formato '$ 2.441.000.000' —
# pd.to_numeric directo los convierte a NaN aunque el dato existe.
# Tras limpiar: cero nulos reales en este CSV.
df["precio_clean"] = (
    df["precio_venta"]
    .astype(str)
    .str.replace(r'[\$\.\s]', '', regex=True)
    .pipe(pd.to_numeric, errors='coerce')
    .apply(lambda v: int(v) if pd.notna(v) and v > 0 else None)
)

print(f"Precio válido   : {df['precio_clean'].notna().sum():,}")
print(f"Nulos reales    : {df['precio_clean'].isna().sum()}")

Precio válido   : 1,240
Nulos reales    : 0


In [15]:
# Dataset transformado completo — sin eliminar duplicados todavía
df_clean = df[[
    "id_propiedad",
    "tipo_inmueble_clean",
    "id_zona",
    "id_estrato",
    "precio_clean",
    "metraje_clean",
    "fecha_clean",
]].copy()

print(f"Registros transformados: {len(df_clean):,}")
print(f"\nNulos por columna:")
print(df_clean.isnull().sum().to_string())

Registros transformados: 1,240

Nulos por columna:
id_propiedad            0
tipo_inmueble_clean     0
id_zona                 0
id_estrato              0
precio_clean            0
metraje_clean          62
fecha_clean             0


## 3. Separación PROPIEDAD / ANUNCIO

El CSV mezcla dos entidades distintas. `propiedad` representa el inmueble físico (atributos estables). `anuncio` representa la publicación comercial (precio y fecha, variables en el tiempo). Cada tabla tiene su propia regla de unicidad.

In [ ]:
# PROPIEDAD — unicidad por id_propiedad
# Atributos físicos estables: zona, tipo, estrato, metraje.
# Los 40 duplicados son exactos — keep='first' descarta la copia.
# id_zona NOT NULL: sin zona la propiedad no es ubicable, se rechaza.
df_propiedad = (
    df_clean
    .drop_duplicates(subset=["id_propiedad"], keep="first")
    [["id_propiedad", "tipo_inmueble_clean", "id_zona", "id_estrato", "metraje_clean"]]
    .reset_index(drop=True)
)

df_propiedad_validas   = df_propiedad[df_propiedad["id_zona"].notna()].reset_index(drop=True)
df_propiedad_rechazadas = df_propiedad[df_propiedad["id_zona"].isna()].reset_index(drop=True)

print(f"Propiedades válidas   : {len(df_propiedad_validas):,}")
print(f"Rechazadas (sin zona) : {len(df_propiedad_rechazadas)}")

Propiedades válidas   : 1,200
Rechazadas (sin zona) : 0


In [ ]:
# ANUNCIO — unicidad por (id_propiedad, fecha_clean)
# precio y fecha son variables — la misma propiedad puede tener múltiples anuncios.
# keep='last': si mismo (id, fecha) llega con precio distinto, el último es el más reciente.
# Sin fecha  no puede participar del índice único, se descarta.
# Sin precio  no es una publicación real, se descarta.
df_anuncio = (
    df_clean[
        df_clean["fecha_clean"].notna() &
        df_clean["precio_clean"].notna()
    ]
    .drop_duplicates(subset=["id_propiedad", "fecha_clean"], keep="last")
    [["id_propiedad", "precio_clean", "fecha_clean"]]
    .reset_index(drop=True)
)

sin_fecha  = df_clean["fecha_clean"].isna().sum()
sin_precio = df_clean["precio_clean"].isna().sum()
print(f"Anuncios a insertar     : {len(df_anuncio):,}")
print(f"Descartados sin fecha   : {sin_fecha}")
print(f"Descartados sin precio  : {sin_precio}")
print(f"\nAnuncios por propiedad:")
print(df_anuncio.groupby('id_propiedad').size().value_counts()
      .rename_axis('anuncios_por_propiedad').to_string())

Anuncios a insertar     : 1,200
Descartados sin fecha   : 0
Descartados sin precio  : 0

Anuncios por propiedad:
anuncios_por_propiedad
1    1200


In [18]:
# VALIDACIÓN FK — simulación de lo que hace la BD en producción
# La FK de anuncio sobre propiedad exige que el id exista en la tabla propiedad.
# En producción la BD lo rechaza sola.
# En el notebook: los anuncios cuyos ids no están en df_propiedad_validas
# pueden existir en la BD de una carga anterior — se reportan como advertencia.
ids_validos = set(df_propiedad_validas["id_propiedad"])

df_anuncio_ok       = df_anuncio[df_anuncio["id_propiedad"].isin(ids_validos)].reset_index(drop=True)
df_anuncio_sin_prop = df_anuncio[~df_anuncio["id_propiedad"].isin(ids_validos)].reset_index(drop=True)

print(f"Anuncios a insertar         : {len(df_anuncio_ok):,}")
print(f"Anuncios sin propiedad (adv): {len(df_anuncio_sin_prop)}")
if len(df_anuncio_sin_prop) > 0:
    print("  → pueden existir en la BD de una carga anterior, no son errores necesariamente")

Anuncios a insertar         : 1,200
Anuncios sin propiedad (adv): 0


## 4. Catálogos maestros

In [ ]:
MUNICIPIOS = {
    1: {"nombre": "Medellín",  "departamento": "Antioquia"},
    2: {"nombre": "Sabaneta",  "departamento": "Antioquia"},
    3: {"nombre": "Envigado",  "departamento": "Antioquia"},
}

ZONAS = {
    1: {"nombre": "El Poblado",        "id_municipio": 1},
    2: {"nombre": "Laureles",           "id_municipio": 1},
    3: {"nombre": "Belén",              "id_municipio": 1},
    4: {"nombre": "Robledo",            "id_municipio": 1},
    5: {"nombre": "Centro",             "id_municipio": 1},
    6: {"nombre": "El Hueco",           "id_municipio": 1},
    7: {"nombre": "Sabaneta",           "id_municipio": 2},
    8: {"nombre": "Envigado",           "id_municipio": 3},
}

TIPOS_INMUEBLE = {
    1: "Casa", 2: "Apartamento", 3: "Apartestudio",
    4: "Finca", 5: "Sin clasificar",
}
TIPO_ID = {v: k for k, v in TIPOS_INMUEBLE.items()}
df_propiedad_validas = df_propiedad_validas.copy()
df_propiedad_validas["id_tipo"] = df_propiedad_validas["tipo_inmueble_clean"].map(TIPO_ID)

ESTRATOS = {
    1: {"numero": 1, "descripcion": "Estrato 1"},
    2: {"numero": 2, "descripcion": "Estrato 2"},
    3: {"numero": 3, "descripcion": "Estrato 3"},
    4: {"numero": 4, "descripcion": "Estrato 4"},
    5: {"numero": 5, "descripcion": "Estrato 5"},
    6: {"numero": 6, "descripcion": "Estrato 6"},
    7: {"numero": 0, "descripcion": "Sin estrato"},
}
print('Catálogos listos')

Catálogos listos


## 5. Carga

> **Modo simulación:** este notebook imprime el SQL que se ejecutaría sin conectarse a ninguna base de datos. Útil para revisar y validar el output antes de correr en producción.

---

En producción el bloque de `print` de cada tabla se reemplaza por esto:

```python
import psycopg2
from psycopg2.extras import execute_batch

conn = psycopg2.connect(
    host=os.getenv('DB_HOST'), dbname=os.getenv('DB_NAME'),
    user=os.getenv('DB_USER'), password=os.getenv('DB_PASSWORD'),
    sslmode='require'
)
cur = conn.cursor()

# El pipeline completo está en etl_propiedades.py
# Se ejecuta diariamente a las 8am via cron:
#   0 8 * * * python3 /ruta/etl_propiedades.py /ruta/propiedades.csv
```

Orden: maestras → propiedad → anuncio. La FK de `anuncio` sobre `propiedad` exige que la propiedad exista antes.

In [25]:
def sql_val(v):
    if v is None or (isinstance(v, float) and math.isnan(v)):
        return 'NULL'
    if isinstance(v, str):
        return "'" + v.replace("'", "''") + "'"
    if isinstance(v, float):
        return f'{v:.2f}'
    return str(v)

print('sql_val listo')

sql_val listo


In [26]:
print('-- TABLAS MAESTRAS')
print()
for id_m, d in MUNICIPIOS.items():
    print(f"INSERT INTO municipio (id_municipio, nombre, departamento) "
          f"VALUES ({id_m}, {sql_val(d['nombre'])}, {sql_val(d['departamento'])}) "
          f"ON CONFLICT DO NOTHING;")
print()
for id_z, d in ZONAS.items():
    print(f"INSERT INTO zona (id_zona, id_municipio, nombre, nombre_normalizado) "
          f"VALUES ({id_z}, {d['id_municipio']}, {sql_val(d['nombre'])}, {sql_val(d['nombre'].lower())}) "
          f"ON CONFLICT DO NOTHING;")
print()
for id_t, desc in TIPOS_INMUEBLE.items():
    print(f"INSERT INTO tipo_inmueble (id_tipo_inmueble, descripcion) "
          f"VALUES ({id_t}, {sql_val(desc)}) ON CONFLICT DO NOTHING;")
print()
for id_e, d in ESTRATOS.items():
    print(f"INSERT INTO estrato (id_estrato, numero, descripcion) "
          f"VALUES ({id_e}, {d['numero']}, {sql_val(d['descripcion'])}) ON CONFLICT DO NOTHING;")

-- TABLAS MAESTRAS

INSERT INTO municipio (id_municipio, nombre, departamento) VALUES (1, 'Medellín', 'Antioquia') ON CONFLICT DO NOTHING;
INSERT INTO municipio (id_municipio, nombre, departamento) VALUES (2, 'Sabaneta', 'Antioquia') ON CONFLICT DO NOTHING;
INSERT INTO municipio (id_municipio, nombre, departamento) VALUES (3, 'Envigado', 'Antioquia') ON CONFLICT DO NOTHING;

INSERT INTO zona (id_zona, id_municipio, nombre, nombre_normalizado) VALUES (1, 1, 'El Poblado', 'el poblado') ON CONFLICT DO NOTHING;
INSERT INTO zona (id_zona, id_municipio, nombre, nombre_normalizado) VALUES (2, 1, 'Laureles', 'laureles') ON CONFLICT DO NOTHING;
INSERT INTO zona (id_zona, id_municipio, nombre, nombre_normalizado) VALUES (3, 1, 'Belén', 'belén') ON CONFLICT DO NOTHING;
INSERT INTO zona (id_zona, id_municipio, nombre, nombre_normalizado) VALUES (4, 1, 'Robledo', 'robledo') ON CONFLICT DO NOTHING;
INSERT INTO zona (id_zona, id_municipio, nombre, nombre_normalizado) VALUES (5, 1, 'Centro', 'centro')

In [27]:
print()
print('-- PROPIEDAD')
print('-- ON CONFLICT (id_propiedad) DO NOTHING')
print()
for _, row in df_propiedad_validas.iterrows():
    print(
        f"INSERT INTO propiedad "
        f"(id_propiedad, id_tipo_inmueble, id_zona, id_estrato, metraje_m2) VALUES ("
        f"{int(row['id_propiedad'])}, "
        f"{int(row['id_tipo'])}, "
        f"{int(row['id_zona'])}, "
        f"{sql_val(None if pd.isna(row['id_estrato']) else int(row['id_estrato']))}, "
        f"{sql_val(row['metraje_clean'])}"
        f") ON CONFLICT (id_propiedad) DO NOTHING;"
    )


-- PROPIEDAD
-- ON CONFLICT (id_propiedad) DO NOTHING

INSERT INTO propiedad (id_propiedad, id_tipo_inmueble, id_zona, id_estrato, metraje_m2) VALUES (681, 1, 7, 2, 249.00) ON CONFLICT (id_propiedad) DO NOTHING;
INSERT INTO propiedad (id_propiedad, id_tipo_inmueble, id_zona, id_estrato, metraje_m2) VALUES (377, 5, 5, 6, 359.00) ON CONFLICT (id_propiedad) DO NOTHING;
INSERT INTO propiedad (id_propiedad, id_tipo_inmueble, id_zona, id_estrato, metraje_m2) VALUES (1084, 1, 1, 4, 227.00) ON CONFLICT (id_propiedad) DO NOTHING;
INSERT INTO propiedad (id_propiedad, id_tipo_inmueble, id_zona, id_estrato, metraje_m2) VALUES (948, 1, 1, 2, 44.00) ON CONFLICT (id_propiedad) DO NOTHING;
INSERT INTO propiedad (id_propiedad, id_tipo_inmueble, id_zona, id_estrato, metraje_m2) VALUES (1017, 5, 4, 4, 353.00) ON CONFLICT (id_propiedad) DO NOTHING;
INSERT INTO propiedad (id_propiedad, id_tipo_inmueble, id_zona, id_estrato, metraje_m2) VALUES (84, 5, 5, 2, 269.00) ON CONFLICT (id_propiedad) DO NOTHING;
IN

In [28]:
print()
print('-- ANUNCIO')
print('-- ON CONFLICT (id_propiedad, fecha_publicacion) DO UPDATE precio')
print()
for _, row in df_anuncio_ok.iterrows():
    print(
        f"INSERT INTO anuncio "
        f"(id_propiedad, precio_venta, fecha_publicacion) VALUES ("
        f"{int(row['id_propiedad'])}, "
        f"{sql_val(row['precio_clean'])}, "
        f"{sql_val(row['fecha_clean'])}"
        f") ON CONFLICT (id_propiedad, fecha_publicacion) "
        f"DO UPDATE SET precio_venta = EXCLUDED.precio_venta;"
    )


-- ANUNCIO
-- ON CONFLICT (id_propiedad, fecha_publicacion) DO UPDATE precio

INSERT INTO anuncio (id_propiedad, precio_venta, fecha_publicacion) VALUES (1084, 822000000, '2023-10-15') ON CONFLICT (id_propiedad, fecha_publicacion) DO UPDATE SET precio_venta = EXCLUDED.precio_venta;
INSERT INTO anuncio (id_propiedad, precio_venta, fecha_publicacion) VALUES (948, 828000000, '2023-10-15') ON CONFLICT (id_propiedad, fecha_publicacion) DO UPDATE SET precio_venta = EXCLUDED.precio_venta;
INSERT INTO anuncio (id_propiedad, precio_venta, fecha_publicacion) VALUES (1017, 1226000000, '2023-10-15') ON CONFLICT (id_propiedad, fecha_publicacion) DO UPDATE SET precio_venta = EXCLUDED.precio_venta;
INSERT INTO anuncio (id_propiedad, precio_venta, fecha_publicacion) VALUES (84, 2009000000, '2023-10-15') ON CONFLICT (id_propiedad, fecha_publicacion) DO UPDATE SET precio_venta = EXCLUDED.precio_venta;
INSERT INTO anuncio (id_propiedad, precio_venta, fecha_publicacion) VALUES (356, 2464000000, '2023-10-

## 6. Reporte de calidad

In [29]:
sin_fecha_vacio   = df_raw["fecha_publicacion"].isna().sum()
sin_fecha_formato = df_clean["fecha_clean"].isna().sum() - sin_fecha_vacio

print(f'Se procesaron {len(df_raw):,} registros del CSV.')
print()
print('Lo que se limpió:')
print(f'  - {df_raw.duplicated().sum()} filas duplicadas eliminadas')
print(f'  - {df_clean["metraje_clean"].isna().sum()} metrajes imposibles (<=1 m²) convertidos a nulo')
print(f'  - {(df_propiedad_validas["tipo_inmueble_clean"] == "Sin clasificar").sum()} propiedades sin tipo → Sin clasificar')
print()
print('Lo que se descartó:')
print(f'  - {len(df_propiedad_rechazadas)} propiedades sin zona (no ubicables)')
print(f'  - {df_clean["precio_clean"].isna().sum()} anuncios sin precio')
print(f'  - {sin_fecha_vacio} anuncios sin fecha')
print()
print('Listo para insertar:')
print(f'  - {len(df_propiedad_validas):,} propiedades')
print(f'  - {len(df_anuncio_ok):,} anuncios')
print()
if sin_fecha_formato > 0:
    print(f'OJO: {sin_fecha_formato} fechas con formato no reconocido — puede que el scraper cambió algo')
if len(df_anuncio_sin_prop) > 0:
    print(f'Aviso: {len(df_anuncio_sin_prop)} anuncios cuya propiedad no llegó en esta carga')
    print('  (pueden existir en BD de carga anterior, la FK lo maneja)')

  REPORTE DE CALIDAD
  Registros raw                : 1,240
  Propiedades válidas          : 1,200
  Propiedades rechazadas       : 0
  Anuncios a insertar          : 1,200
  Anuncios sin propiedad (adv) : 0
-------------------------------------------------------
  Duplicados eliminados        : 40
  Metraje → NULL (<= 1)        : 62
  Sin fecha (campo vacío)      : 0
  Sin fecha (fmt desconocido)  : 0
  Tipo Sin clasificar          : 237
  Propiedades con >1 anuncio   : 0
